# Übertragungstest: Gilt das Warnzeichen auch an anderen Druckern und 2026?
**Internet of Production – KI, Woche 3, Block 5** · etwa 30 min · Zweiergruppen

**Roter Faden:** In Woche 1 und 2 haben Sie das Warnzeichen gefunden (die Energie je Minute steigt vor einer Extruder-Blockade) und an **einem** Drucker in **2024/25** geprüft. Der Monitor aus Block 6 läuft an **anderen Druckern** und mit **neuen Daten**. Bevor wir ihn bauen, fragen wir: Überträgt sich das Modell?

**Aufbau**
- **Training (eingefroren):** KNN (k = 25) und die z-Wert-Regel, beide an den ersten 70 % von Drucker 1 aus 2024/25 angepasst (wie in Woche 1 und Übung 2b). Sie werden **nicht** neu trainiert.
- **Test:** die sechs Drucker des Jahres 2026 (5. Jan bis 19. Sep 2026). Die Farm wurde um drei Drucker erweitert.
- **Frage:** Schlägt ein eingefrorenes Modell eine Grenze, die für jeden Drucker neu gesetzt wird (*Baseline je Drucker*)?

**Notieren Sie Ihre Erwartung, bevor Sie etwas ausführen:** Was erwarten Sie für AUC, Recall und Fehlalarme an den neuen Druckern?

**Ihre Erwartung (ein bis zwei Sätze):**

AUC etwas unter 0,88 (andere Drucker, anderes Jahr), Recall etwa 50 bis 60 %, ähnlich viele Fehlalarme. Die einfache Regel könnte besser standhalten, weil sie nur ein Merkmal nutzt.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, recall_score, precision_score

DATA_OLD = Path('../../00_Data/printfarm')          # 2024/25, printers 0-2: used for TRAINING (as in weeks 1 and 2)
DATA_NEW = Path('../../00_Data/printfarm_2026')     # 2026, printers 0-5: used for TESTING
HORIZON = 360                                       # minutes
FEATURES = ['E30', 'E120', 'slope']

def make_blockage_data(folder, k, ts_format=None, horizon=HORIZON):
    """The pipeline of exercise 2 in one function: one row per 10th printing minute. ts_format=None: ISO time stamps (2026)."""
    d = pd.read_csv(folder / f'Printer_{k}.csv', sep=';', decimal=',', low_memory=False).dropna(how='all')
    d['Timestamp'] = pd.to_datetime(d['Timestamp'], format=ts_format)
    d = d.set_index('Timestamp')
    d['Energy'] = d['Total_Power'].diff()
    e = pd.read_csv(folder / f'Printer_{k}_error.csv', sep=';').dropna(how='all')
    e['Timestamp'] = pd.to_datetime(e['Timestamp'], format=ts_format)
    blockages = pd.to_datetime(np.sort(e.loc[e['Error'] == 'Extruder blocked', 'Timestamp'].values))
    p = d[d['State'] == 'printing'].copy()
    p['E'] = p['Energy'] * 1000                              # Wh per minute
    idx = p.index.values
    nxt = np.searchsorted(blockages.values, idx)
    mtb = np.where(nxt < len(blockages), (blockages.values[np.minimum(nxt, len(blockages) - 1)] - idx) / np.timedelta64(1, 'm'), np.inf)
    p['y'] = ((mtb > 0) & (mtb <= horizon)).astype(int)      # only used to EVALUATE
    p['E30'] = p['E'].rolling(30).mean()
    p['E120'] = p['E'].rolling(120).mean()
    p['slope'] = p['E120'] - p['E120'].shift(240)
    return p.dropna(subset=FEATURES).iloc[::10], blockages

def evaluate_alarms(alarm, test, blockages):
    """alarm: boolean Series on the index of `test`. Blockages announced within 6 h and false alarm episodes per week."""
    t0, t1 = test.index[0], test.index[-1]
    blk = [t for t in blockages if t0 <= t <= t1]
    announced = sum(bool(alarm[(alarm.index >= t - pd.Timedelta(hours=6)) & (alarm.index <= t)].any()) for t in blk)
    times = alarm[alarm].index
    weeks = (t1 - t0).days / 7
    if len(times) == 0:
        return announced, len(blk), 0.0
    new_episode = np.concatenate([[0], np.diff(times.values) > np.timedelta64(6, 'h')])   # a gap of more than 6 h starts a new episode
    ep = pd.DataFrame({'ep': np.cumsum(new_episode), 'y': test.loc[times, 'y'].values}).groupby('ep')['y'].max()
    return announced, len(blk), float((ep == 0).sum() / weeks)

def wilson(k, n, z=1.96):
    """95 % interval of a share k / n (Wilson)."""
    p = k / n
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centre - half, centre + half

def score_row(printer, model, test, blockages, score, alarm):
    ann, n_blk, fa = evaluate_alarms(alarm, test, blockages)
    return {'printer': printer, 'model': model, 'AUC': roc_auc_score(test['y'], score), 'recall': recall_score(test['y'], alarm),
            'precision': precision_score(test['y'], alarm, zero_division=0), 'announced': ann, 'blockages': n_blk, 'false episodes / week': fa}

## Schritt 1: Referenzmodelle einfrieren
Passen Sie KNN und die z-Wert-Grenze an den ersten 70 % von Drucker 1 aus 2024/25 an. Ab hier werden sie nicht mehr verändert.

In [ ]:
old, old_blockages = make_blockage_data(DATA_OLD, 1, ts_format='%d.%m.%Y %H:%M')
reference_old = old.iloc[:int(len(old) * 0.7)]

knn_frozen = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=25)).fit(reference_old[FEATURES], reference_old['y'])
mu_old, sd_old = reference_old['E120'].mean(), reference_old['E120'].std()
z_limit_old = np.quantile((reference_old['E120'] - mu_old) / sd_old, 0.95)
print(f'normal E120 (2024/25, printer 1): {mu_old:.3f} +/- {sd_old:.3f} Wh/min; limit at z = {z_limit_old:.2f}, i.e. E120 > {mu_old + z_limit_old * sd_old:.3f}')

## Schritt 2: Eingefrorene Modelle auf die sechs Drucker von 2026 anwenden
Ganzes Jahr, ohne Nachtraining. Je Drucker: AUC, Recall, innerhalb von 6 h angekündigte Blockaden und Fehlalarm-Episoden je Woche. Der Anteil angekündigter Blockaden erhält ein 95-%-Intervall (Wilson), weil er nur auf 40 bis 60 Ereignissen beruht.

In [ ]:
printers = {k: make_blockage_data(DATA_NEW, k) for k in range(6)}

rows = []
for k, (d, blk) in printers.items():
    z = (d['E120'] - mu_old) / sd_old
    rows.append(score_row(k, 'KNN (frozen)', d, blk, knn_frozen.predict_proba(d[FEATURES])[:, 1], pd.Series(knn_frozen.predict(d[FEATURES]) == 1, index=d.index)))
    rows.append(score_row(k, 'z-value rule (frozen)', d, blk, z, z > z_limit_old))
res_all = pd.DataFrame(rows)
res_all['announced share'] = res_all['announced'] / res_all['blockages']
res_all[['ci low', 'ci high']] = [wilson(a, n) for a, n in zip(res_all['announced'], res_all['blockages'])]
display(res_all.round(3).set_index(['printer', 'model']))
print(res_all.groupby('model')[['AUC', 'recall', 'precision', 'false episodes / week']].mean().round(3))

**Überträgt sich das eingefrorene Modell? Vergleichen Sie mit Ihrer Erwartung. Welches der beiden Modelle hält besser, und bei welcher Kennzahl?**

Ja. An allen sechs Druckern bleibt die AUC zwischen etwa 0,86 und 0,94, im Bereich des Trainingsdruckers in 2024/25 (0,88 bis 0,91), und 38 bis 51 der 42 bis 57 Blockaden je Drucker werden angekündigt. Die z-Wert-Regel hat die höhere AUC (0,89 bis 0,94 gegenüber 0,86 bis 0,89 beim KNN) und den höheren Recall (64 bis 74 % gegenüber 50 bis 61 %), bei etwas geringerer Precision (61 bis 77 % gegenüber 68 bis 88 %). Die einfache Regel ist mindestens so gut wie das KNN, wie in Woche 2.

## Schritt 3: Hat sich das Normalniveau verschoben?
Ein eingefrorenes Modell nimmt an, dass sich „normal“ nicht geändert hat. Vergleichen Sie die Energie je Minute der Drucker von 2026 mit der Referenz von 2024/25.

In [ ]:
drift = pd.DataFrame({k: {'mean E120': d['E120'].mean(), 'sd E120': d['E120'].std()} for k, (d, _) in printers.items()}).T
drift.loc['2024/25 reference'] = [mu_old, sd_old]
display(drift.round(4))

fig, ax = plt.subplots(figsize=(9, 3.6))
for k, (d, _) in printers.items():
    normal = d[d['y'] == 0]['E120'].resample('MS').mean()
    ax.plot(normal.index, normal.values, marker='o', label=f'printer {k}')
ax.axhline(mu_old, color='k', ls='--', label='reference 2024/25')
ax.set_ylabel('mean E120 of normal operation (Wh/min)'); ax.legend(ncol=4, fontsize=8); plt.show()

**Gibt es Drift? Was würden Sie tun, wenn das Niveau 2026 um 3 % über der Referenz läge?**

Die mittlere E120 liegt bei allen Druckern innerhalb von 0,005 Wh/min um die Referenz (1,645 gegenüber 1,647 bis 1,650); die Streuung ist etwas größer (0,019 bis 0,023 gegenüber 0,017), am meisten bei Drucker 5. In der Abbildung ist kein starker Trend über die Monate zu erkennen. Bei einem um 3 % höheren Niveau (etwa +0,05 Wh/min, drei Standardabweichungen) würde die eingefrorene Grenze fast dauernd alarmieren. Dann muss die Referenz je Drucker neu gelernt werden (Schritt 4) oder nach jeder Wartung, und der Monitor braucht eine Drift-Prüfung.

## Schritt 4: Die Baseline je Drucker
Lernen Sie „normal“ für jeden Drucker neu aus seinen **eigenen** ersten 70 % von 2026 und testen Sie auf den letzten 30 %. Alle vier Modelle werden auf demselben Testzeitraum bewertet, damit sie vergleichbar sind.

In [ ]:
rows = []
for k, (d, blk) in printers.items():
    cut = int(len(d) * 0.7)
    ref, test = d.iloc[:cut], d.iloc[cut:]
    mu, sd = ref['E120'].mean(), ref['E120'].std()
    z_ref, z_test = (ref['E120'] - mu) / sd, (test['E120'] - mu) / sd
    limit = np.quantile(z_ref, 0.95)
    knn_own = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=25)).fit(ref[FEATURES], ref['y'])
    z_frozen = (test['E120'] - mu_old) / sd_old
    rows += [
        score_row(k, '1 z-value rule, frozen (2024/25)', test, blk, z_frozen, z_frozen > z_limit_old),
        score_row(k, '2 z-value rule, own reference', test, blk, z_test, z_test > limit),
        score_row(k, '3 KNN, frozen (2024/25)', test, blk, knn_frozen.predict_proba(test[FEATURES])[:, 1], pd.Series(knn_frozen.predict(test[FEATURES]) == 1, index=test.index)),
        score_row(k, '4 KNN, own reference', test, blk, knn_own.predict_proba(test[FEATURES])[:, 1], pd.Series(knn_own.predict(test[FEATURES]) == 1, index=test.index)),
    ]
res = pd.DataFrame(rows)
summary = res.groupby('model').agg(AUC=('AUC', 'mean'), recall=('recall', 'mean'), precision=('precision', 'mean'),
                                   announced=('announced', 'sum'), blockages=('blockages', 'sum'), false_per_week=('false episodes / week', 'mean'))
summary['announced share'] = summary['announced'] / summary['blockages']
display(summary.round(3))

**Schlägt die Baseline je Drucker das eingefrorene Modell? Betrachten Sie AUC, den Anteil angekündigter Blockaden und die Fehlalarme. Was heißt das für den Monitor?**

Auf demselben Testzeitraum (letzte 30 % von 2026, sechs Drucker, 73 Blockaden) ist die AUC der z-Wert-Regel mit eingefrorener oder eigener Referenz gleich (0,937), weil der z-Wert nur eine lineare Transformation von E120 ist: Eine Baseline je Drucker verschiebt, wo die Grenze liegt, nicht wie gut der Score ordnet. Eingefrorene Grenze: 68 von 73 Blockaden angekündigt (93 %), 0,64 Fehlalarm-Episoden je Drucker und Woche; eigene Grenze: 64 von 73 (88 %), 0,38 je Woche. Das ist ein anderer Zielkonflikt, kein besseres Modell, und der Unterschied von 4 Blockaden liegt innerhalb der Unsicherheit. Das KNN mit eigener Referenz (AUC 0,93) ist besser als das eingefrorene (0,90), verursacht aber die meisten Fehlalarme (0,90 je Woche). Für den Monitor: mit der eingefrorenen Regel beginnen, die Referenz je Drucker als Option vorsehen und die Drift beobachten (Schritt 3).

## Schritt 5: Wie sicher ist das?
Fassen Sie die sechs Drucker zusammen und geben Sie den angekündigten Anteil der eingefrorenen z-Wert-Regel mit Intervall an. Die Drucker gehören zur selben Farm und zum selben Prozess, sind also **keine** sechs unabhängigen Stichproben.

In [ ]:
tot = res_all[res_all['model'] == 'z-value rule (frozen)'][['announced', 'blockages']].sum()
lo, hi = wilson(tot['announced'], tot['blockages'])
print(f"frozen z-value rule, whole of 2026: {int(tot['announced'])} of {int(tot['blockages'])} blockages announced = {tot['announced'] / tot['blockages']:.0%} (95 % interval {lo:.0%} to {hi:.0%})")
print('false alarm episodes per printer and week:', res_all[res_all['model'] == 'z-value rule (frozen)']['false episodes / week'].round(2).tolist())

# Chance level: shift the alarm series of every printer by a random time (the alarm rate stays the same, the link to the blockages is destroyed)
def announced_fast(alarm_times, blockages):
    t = np.sort(alarm_times.values)
    return sum(np.searchsorted(t, b, 'right') > np.searchsorted(t, b - np.timedelta64(6, 'h'), 'left') for b in blockages.values)

rng = np.random.default_rng(0)
null = []
for _ in range(200):
    n_ann = n_blk = 0
    for k, (d, blk) in printers.items():
        alarm = np.roll(((d['E120'] - mu_old) / sd_old > z_limit_old).values, rng.integers(500, len(d) - 500))
        b = blk[(blk >= d.index[0]) & (blk <= d.index[-1])]
        n_ann += announced_fast(d.index[alarm], b); n_blk += len(b)
    null.append(n_ann / n_blk)
print(f'chance level with shifted alarms: {np.mean(null):.0%} (5 to 95 %: {np.quantile(null, 0.05):.0%} to {np.quantile(null, 0.95):.0%}) against {tot["announced"] / tot["blockages"]:.0%} observed')

**Was können Sie über die Übertragung behaupten, was nicht? Welchen Schwellwert geben Sie dem Monitor, und wer entscheidet das?**

Behaupten: Die eingefrorene Regel kündigt etwa 91 % der 295 Blockaden von 2026 an (Intervall 87 bis 94 %), bei etwa 0,4 bis 0,8 Fehlalarm-Episoden je Drucker und Woche, gegenüber einem Zufallsniveau von etwa 19 % (15 bis 23 %) bei Alarmen gleicher Häufigkeit: Das Warnzeichen überträgt sich ohne Nachtraining auf neue Drucker und ein neues Jahr, und das Energieniveau hat sich nicht verschoben. Nicht behaupten: dass die Drucker unabhängige Stichproben sind (gleiche Farm, gleicher Prozess), dass der angekündigte Anteil ein Recall ist (jeder Alarm in den 6 h zählt, der Recall je Stichprobe beträgt 64 bis 74 %) oder dass die Regel nach einer Wartung oder bei anderem Produkt weiter funktioniert. Der Schwellwert ist das Kostenverhältnis zwischen einer Prüfung (10 Minuten) und einer Blockade (etwa 12 Stunden) und wird vom Vorfertigungsleiter entschieden, nicht vom Modell; für den Monitor wird die Grenze ein Parameter (z = 1,79 als Vorgabe).